In [7]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch

In [8]:
dataset_path = "../../bats_transformer/data/2022_barn_daytime_2secs/splits"
dataset_path = "../../bats_transformer/data/2022_barn_2secs_myca/splits"

In [9]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [14]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": dataset_path,
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=16,
    workers=4
)

In [15]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [16]:
means = np.zeros(32)
num_batches = 0
for batch in tqdm(train_data):
    x_t, x_c, y_t, y_c = batch
    # print(y_c.shape)
    # print(y_c.squeeze().mean(dim=0).numpy())
    means += y_c.squeeze().mean(dim=0).numpy()
    num_batches += (y_c.shape[0] / 64)
    # print(means)
means /= num_batches

  0%|          | 4/12258 [00:07<6:06:03,  1.79s/it] 


RuntimeError: DataLoader worker (pid(s) 14749) exited unexpectedly

In [ ]:
means

array([ 0.72608796,  0.08561341,  0.15239749,  0.19423689,  0.09717899,
        0.09431835,  0.11973082,  0.01940801,  0.17696234,  0.13043439,
        0.11122072,  0.05431392,  0.11090818, -4.64952683,  0.10659878,
        0.10390843,  0.14142229,  0.05490304,  0.11113161,  0.11492797,
        0.09493767,  0.10428831,  0.03260223, -0.10773394, -0.05839958,
       -0.01220693, -0.03017957,  0.17747218, -0.04799524,  0.0896759 ,
       -0.14026251,  0.04089851])

In [ ]:
pred = means

In [ ]:
truths = []
errors = []
preds = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    for row in y_c:
        truths.append(row.numpy()[0])
        preds.append(pred)
        errors.append((row - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 401/401 [01:31<00:00,  4.37it/s]


In [ ]:
pd.DataFrame(preds)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
1,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
2,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
3,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
4,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12799,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
12800,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
12801,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899
12802,0.726088,0.085613,0.152397,0.194237,0.097179,0.094318,0.119731,0.019408,0.176962,0.130434,...,0.032602,-0.107734,-0.0584,-0.012207,-0.03018,0.177472,-0.047995,0.089676,-0.140263,0.040899


In [ ]:
target_columns = train_data.dataset.target_cols
target_columns

['TimeInFile',
 'PrecedingIntrvl',
 'HiFreq',
 'Bndwdth',
 'FreqMaxPwr',
 'PrcntMaxAmpDur',
 'FreqKnee',
 'PrcntKneeDur',
 'StartF',
 'UpprKnFreq',
 'HiFtoUpprKnAmp',
 'HiFtoKnAmp',
 'HiFtoFcAmp',
 'UpprKnToKnAmp',
 'KnToFcAmp',
 'LdgToFcAmp',
 'FreqCtr',
 'FFwd32dB',
 'FFwd20dB',
 'FFwd15dB',
 'FBak5dB',
 'FFwd5dB',
 'Bndw32dB',
 'Amp1stQrtl',
 'Amp2ndQrtl',
 'Amp3rdQrtl',
 'Amp4thQrtl',
 '1st10kHzSlp',
 '1st5to15kHzSlp',
 '1st10kHzExp',
 '1st5to15kHzExp',
 'AmpK@start']

In [ ]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [ ]:
mse

array([ 0.77067132,  0.81240754,  1.12092984,  1.13300603,  1.0409205 ,
        1.00219221,  1.05666838,  0.93500859,  1.11267954,  1.08254684,
        1.08064152,  1.22327714,  1.03371523, 15.02136517,  1.01642358,
        1.04679565,  0.992202  ,  1.0826606 ,  1.05599683,  1.05297731,
        1.00898385,  1.01296089,  1.20032264,  1.14693253,  1.15081298,
        1.05177259,  0.99270428,  1.14947555,  2.3892802 ,  1.06662058,
        2.25722142,  0.90052184])

In [ ]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile          0.770671
PrecedingIntrvl     0.812408
HiFreq              1.120930
Bndwdth             1.133006
FreqMaxPwr          1.040920
PrcntMaxAmpDur      1.002192
FreqKnee            1.056668
PrcntKneeDur        0.935009
StartF              1.112680
UpprKnFreq          1.082547
HiFtoUpprKnAmp      1.080642
HiFtoKnAmp          1.223277
HiFtoFcAmp          1.033715
UpprKnToKnAmp      15.021365
KnToFcAmp           1.016424
LdgToFcAmp          1.046796
FreqCtr             0.992202
FFwd32dB            1.082661
FFwd20dB            1.055997
FFwd15dB            1.052977
FBak5dB             1.008984
FFwd5dB             1.012961
Bndw32dB            1.200323
Amp1stQrtl          1.146933
Amp2ndQrtl          1.150813
Amp3rdQrtl          1.051773
Amp4thQrtl          0.992704
1st10kHzSlp         1.149476
1st5to15kHzSlp      2.389280
1st10kHzExp         1.066621
1st5to15kHzExp      2.257221
AmpK@start          0.900522
dtype: float64

In [ ]:
mse

array([ 0.77067132,  0.81240754,  1.12092984,  1.13300603,  1.0409205 ,
        1.00219221,  1.05666838,  0.93500859,  1.11267954,  1.08254684,
        1.08064152,  1.22327714,  1.03371523, 15.02136517,  1.01642358,
        1.04679565,  0.992202  ,  1.0826606 ,  1.05599683,  1.05297731,
        1.00898385,  1.01296089,  1.20032264,  1.14693253,  1.15081298,
        1.05177259,  0.99270428,  1.14947555,  2.3892802 ,  1.06662058,
        2.25722142,  0.90052184])

In [ ]:
errors.mean(axis=0)

array([-3.72417448e-01, -9.38708455e-02, -1.78922872e-01, -1.80616030e-01,
       -1.78538927e-01, -1.50951903e-01, -1.50281128e-01, -5.92302329e-02,
       -1.90981680e-01, -1.82476241e-01, -1.39343220e-01, -1.20489502e-01,
       -1.74178970e-01,  2.43601915e+00, -1.61737324e-01, -1.35646963e-01,
       -2.11015620e-01, -1.71536322e-01, -2.24908204e-01, -2.22663585e-01,
       -1.76948179e-01, -2.07775961e-01, -1.46269786e-01,  7.73755890e-02,
        1.21982869e-01,  1.23300803e-01,  6.35412143e-02, -1.57720753e-01,
       -1.32918341e-01, -1.25210916e-01, -9.65402201e-02, -2.44189079e-05])

In [ ]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

1.5625217238846945 1.1283654837811448
